# Protocol Buffers и gRPC на практике

**Дисциплина:** Программирование на Python · Практика к лекции 2 «Микросервисы. Проектирование веб-сервисов»

Теория — в конспекте «Protocol Buffers и gRPC». В этом ноутбуке мы шаг за шагом:

1. Опишем данные в `.proto`, сгенерируем классы и посмотрим, как protobuf превращает объекты в байты.
2. Разберём двоичный формат вручную и проверим правила совместимости схем.
3. Соберём первый gRPC-сервис Hello World и расширим его новым методом.
4. Реализуем три вида потоковых вызовов.
5. Научимся работать с ошибками, дедлайнами, метаданными и асинхронным клиентом.
6. Соберём два микросервиса: Recommendations (gRPC) и Marketplace (FastAPI + REST), запустим их локально и в Docker Compose.

**Окружение:** Linux или macOS, Python 3.10+. Все ячейки выполняются по порядку. Файлы проекта создаются командой `%%writefile` в каталоге `grpc_practice/` рядом с ноутбуком.

> **Важно.** Если вы изменили `.proto` и перегенерировали код, **перезапустите ядро** (Kernel → Restart) и выполните ячейки заново: Python не перезагружает уже импортированные модули.

## 0. Подготовка окружения

Работаем в виртуальном окружении. В терминале, в каталоге с ноутбуком:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install jupyter
jupyter notebook
```

Остальные зависимости ставим из ноутбука. Команда `%pip` устанавливает пакеты именно в то окружение, в котором работает ядро Jupyter (в отличие от `!pip`, который может обратиться к другому Python).

| Пакет | Назначение |
|---|---|
| `grpcio` | Библиотека gRPC: клиент и сервер |
| `grpcio-tools` | Компилятор `protoc` и плагин генерации кода для Python; тянет за собой `protobuf` |
| `grpcio-reflection` | Server reflection для инструментов отладки |
| `fastapi[standard]` | Веб-фреймворк и сервер `uvicorn` для части 5 |
| `pytest`, `requests` | Тесты и HTTP-запросы из ноутбука |

In [1]:
%pip install -q grpcio grpcio-tools grpcio-reflection "fastapi[standard]" pytest requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.6/807.6 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.8/354.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/108.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 928.0/928.0 kB 47.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.


In [2]:
import os
import sys
from pathlib import Path

# Рабочий каталог практики. Повторный запуск ячейки безопасен.
if Path.cwd().name != "grpc_practice":
    Path("grpc_practice").mkdir(exist_ok=True)
    os.chdir("grpc_practice")

# Каталог проекта — в начало пути поиска модулей, чтобы импортировать сгенерированный код
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

for directory in ("protos/library/v1", "protos/evolution/v1", "protos/evolution/v2",
                  "microservices/protos", "microservices/recommendations", "microservices/marketplace"):
    Path(directory).mkdir(parents=True, exist_ok=True)

import grpc
import google.protobuf

print("Рабочий каталог:", Path.cwd())
print("grpcio:", grpc.__version__, "| protobuf:", google.protobuf.__version__)

Рабочий каталог: /content/grpc_practice
grpcio: 1.84.0 | protobuf: 7.36.2


---
# Часть 1. Protocol Buffers без gRPC

Protobuf — только формат сериализации. Сначала поработаем с ним отдельно от сети: опишем данные, сгенерируем классы, сериализуем и разберём сообщения.

## 1.1. Описываем данные

Файл кладём по пути `protos/library/v1/library.proto` — путь повторяет пакет `library.v1`. Тогда модуль Python будет называться `library.v1.library_pb2`: в Python путь модуля определяется **расположением файла относительно `-I`**, а не директивой `package`.

In [ ]:
%%writefile protos/library/v1/library.proto
syntax = "proto3";

package library.v1;

import "google/protobuf/timestamp.proto";

// Жанр книги. Нулевое значение — «не задано».
enum Genre {
  GENRE_UNSPECIFIED = 0;
  GENRE_FICTION = 1;
  GENRE_SCIENCE = 2;
  GENRE_HISTORY = 3;
}

message Author {
  string name = 1;
  optional int32 birth_year = 2;   // optional: можно отличить «не задано» от 0
}

message Book {
  int64 id = 1;
  string title = 2;
  repeated Author authors = 3;                   // список вложенных сообщений
  Genre genre = 4;
  map<string, string> attributes = 5;            // словарь
  google.protobuf.Timestamp published_at = 6;    // стандартный тип «момент времени»
  oneof source {                                 // задано не более одного поля
    string isbn = 7;
    string ebook_url = 8;
  }
}

message Catalog {
  repeated Book books = 1;
}

## 1.2. Генерируем код

Компилятор вызываем как модуль `grpc_tools.protoc`. Через `{sys.executable}` запускаем тот же Python, в котором работает ядро.

- `-I protos` — корень для поиска `.proto` и импортов;
- `--python_out=.` — классы сообщений (`*_pb2.py`);
- `--pyi_out=.` — аннотации типов для IDE (`*_pb2.pyi`).

Файл `google/protobuf/timestamp.proto` искать не нужно: `grpc_tools.protoc` сам подключает стандартные типы.

In [ ]:
!{sys.executable} -m grpc_tools.protoc -I protos --python_out=. --pyi_out=. protos/library/v1/library.proto
!ls library/v1

## 1.3. Создаём сообщение

Сгенерированный класс ведёт себя как обычный объект. Особенности:

- у `repeated`-поля сообщений есть метод `add()`, создающий новый элемент;
- `map`-поле работает как словарь;
- поле-сообщение (`published_at`) нельзя присвоить целиком, его изменяют «на месте».

In [ ]:
from datetime import datetime, timezone

from library.v1 import library_pb2

book = library_pb2.Book(id=1, title="Solaris", genre=library_pb2.GENRE_FICTION)
book.authors.add(name="Stanisław Lem", birth_year=1921)
book.attributes["language"] = "pl"
book.attributes["publisher"] = "MON"
book.published_at.FromDatetime(datetime(1961, 1, 1, tzinfo=timezone.utc))
book.isbn = "978-0-15-602760-1"

print(book)   # текстовое представление для отладки

## 1.4. Сериализация и разбор

`SerializeToString()` возвращает `bytes` (название историческое — строк там нет). `FromString()` создаёт объект из байтов.

In [ ]:
data = book.SerializeToString()
print(type(data).__name__, len(data), "байт")
print(data.hex(" "))

restored = library_pb2.Book.FromString(data)
print("Объекты равны:", restored == book)
print("Заголовок после разбора:", restored.title)

## 1.5. Сравнение с JSON

Модуль `json_format` переводит сообщение в JSON по стандартным правилам protobuf. Обратите внимание:

- имена полей по умолчанию в lowerCamelCase (`publishedAt`);
- 64-битный `id` записан **строкой**: JavaScript не умеет точно хранить такие числа;
- перечисление записано именем, `Timestamp` — строкой RFC 3339.

In [ ]:
import json

from google.protobuf import json_format

as_json = json_format.MessageToJson(book, indent=None)
print(as_json)
print()
print("JSON:    ", len(as_json.encode()), "байт")
print("protobuf:", len(data), "байт")

# Имена полей как в .proto
print(json_format.MessageToDict(book, preserving_proto_field_name=True)["published_at"])

На одном сообщении разница невелика. Сравним каталог из 1000 книг — и по размеру, и по скорости. Для честности JSON сериализуем стандартным модулем `json` из обычных словарей.

In [ ]:
import timeit

catalog = library_pb2.Catalog()
for i in range(1, 1001):
    item = catalog.books.add(id=i, title=f"Book #{i}", genre=library_pb2.GENRE_SCIENCE)
    item.authors.add(name=f"Author {i % 50}", birth_year=1900 + i % 100)
    item.attributes["language"] = "en"

proto_bytes = catalog.SerializeToString()
catalog_dict = json_format.MessageToDict(catalog, preserving_proto_field_name=True)
json_bytes = json.dumps(catalog_dict).encode()

print(f"protobuf: {len(proto_bytes):>7} байт")
print(f"JSON:     {len(json_bytes):>7} байт  (в {len(json_bytes) / len(proto_bytes):.1f} раза больше)")

n = 200
t_proto_dump = timeit.timeit(catalog.SerializeToString, number=n) / n * 1000
t_json_dump = timeit.timeit(lambda: json.dumps(catalog_dict), number=n) / n * 1000
t_proto_load = timeit.timeit(lambda: library_pb2.Catalog.FromString(proto_bytes), number=n) / n * 1000
t_json_load = timeit.timeit(lambda: json.loads(json_bytes), number=n) / n * 1000
print(f"\nСериализация: protobuf {t_proto_dump:.3f} мс, json {t_json_dump:.3f} мс")
print(f"Разбор:        protobuf {t_proto_load:.3f} мс, json {t_json_load:.3f} мс")

Цифры зависят от компьютера и данных, но порядок разницы в размере сохраняется: в protobuf нет имён полей, а числа кодируются компактно. Скорость в Python определяется тем, что обе библиотеки написаны на C; выигрыш protobuf заметнее в языках со статической типизацией и на больших вложенных структурах.

## 1.6. Значения по умолчанию и наличие поля

В proto3 у скалярных полей нет `null`: незаданное поле возвращает значение по умолчанию и не попадает в сериализованные байты.

In [ ]:
empty = library_pb2.Book()
print(repr(empty.title), empty.id, empty.genre, list(empty.authors), dict(empty.attributes))
print("Пустое сообщение занимает", len(empty.SerializeToString()), "байт")

# Поле-сообщение отслеживает наличие всегда
print("published_at задано?", empty.HasField("published_at"))

# optional-скаляр тоже отслеживает наличие
author = library_pb2.Author(name="Lem")
print("birth_year задано?", author.HasField("birth_year"))
author.birth_year = 0
print("birth_year задано после присваивания 0?", author.HasField("birth_year"))

# Обычный скаляр наличие не отслеживает
try:
    empty.HasField("title")
except ValueError as error:
    print("ValueError:", error)

## 1.7. oneof и проверка типов

In [ ]:
b = library_pb2.Book(title="Solaris", isbn="978-0-15-602760-1")
print("Задано:", b.WhichOneof("source"))
b.ebook_url = "https://example.com/solaris.epub"
print("Задано:", b.WhichOneof("source"), "| isbn теперь:", repr(b.isbn))

# Типы проверяются при присваивании
for value in ("1", 2**70, 1.5):
    try:
        b.id = value
    except (TypeError, ValueError) as error:
        print(f"{value!r:>25} → {type(error).__name__}: {error}")

# Поле-сообщение нельзя присвоить целиком
from google.protobuf.timestamp_pb2 import Timestamp
try:
    b.published_at = Timestamp()
except AttributeError as error:
    print("AttributeError:", error)
b.published_at.CopyFrom(Timestamp(seconds=0))   # так правильно
print(b.published_at.ToDatetime())

---
# Часть 2. Как устроены байты

## 2.1. Разбираем сообщение вручную

Сообщение protobuf — последовательность пар «ключ — значение». Ключ — число `(номер_поля << 3) | wire_type`. Напишем небольшой разборщик, который не знает схему и видит только номера полей и типы кодирования.

In [ ]:
WIRE_TYPES = {0: "VARINT", 1: "I64", 2: "LEN", 5: "I32"}


def read_varint(data: bytes, pos: int) -> tuple[int, int]:
    """Читает varint начиная с позиции pos. Возвращает (значение, новая позиция)."""
    result = shift = 0
    while True:
        byte = data[pos]
        pos += 1
        result |= (byte & 0x7F) << shift     # младшие 7 бит — полезные данные
        if not byte & 0x80:                   # старший бит 0 — последний байт числа
            return result, pos
        shift += 7


def dump_fields(data: bytes, indent: str = "") -> None:
    """Печатает поля сообщения без знания схемы."""
    pos = 0
    while pos < len(data):
        key, pos = read_varint(data, pos)
        field_number, wire_type = key >> 3, key & 0b111
        if wire_type == 0:
            value, pos = read_varint(data, pos)
        elif wire_type == 1:
            value, pos = data[pos:pos + 8], pos + 8
        elif wire_type == 2:
            length, pos = read_varint(data, pos)
            value, pos = data[pos:pos + length], pos + length
        elif wire_type == 5:
            value, pos = data[pos:pos + 4], pos + 4
        else:
            raise ValueError(f"Неподдерживаемый wire type {wire_type}")
        print(f"{indent}поле {field_number:>2}  {WIRE_TYPES[wire_type]:<6}  {value!r}")


dump_fields(book.SerializeToString())

Сравните вывод с `.proto`: поле 1 — `id`, поле 3 — автор (вложенное сообщение закодировано как `LEN`), поле 5 встречается дважды — каждый элемент `map` хранится как отдельная запись, поле 6 — `Timestamp`. Разберём вложенное сообщение автора тем же разборщиком:

In [ ]:
dump_fields(book.authors[0].SerializeToString(), indent="  ")

## 2.2. Varint, zigzag и упакованные списки

Опишем маленькое сообщение и посмотрим на байты разных значений. Классический пример из документации: `a = 150` кодируется как `08 96 01`.

In [ ]:
%%writefile protos/encoding_demo.proto
syntax = "proto3";

package encoding_demo;

message Demo {
  int32 a = 1;
  string b = 2;
  int32 negative = 3;     // отрицательное число в int32
  sint32 zigzag = 4;      // то же, но с zigzag-кодированием
  repeated int32 nums = 5;
}

In [ ]:
!{sys.executable} -m grpc_tools.protoc -I protos --python_out=. --pyi_out=. protos/encoding_demo.proto

import encoding_demo_pb2 as ed

examples = {
    "a = 150": ed.Demo(a=150),
    "b = 'testing'": ed.Demo(b="testing"),
    "negative = -1 (int32)": ed.Demo(negative=-1),
    "zigzag = -1 (sint32)": ed.Demo(zigzag=-1),
    "zigzag = 1 (sint32)": ed.Demo(zigzag=1),
    "nums = [1, 2, 300]": ed.Demo(nums=[1, 2, 300]),
}
for title, message in examples.items():
    raw = message.SerializeToString()
    print(f"{title:<24} {len(raw):>2} байт: {raw.hex(' ')}")

Что видно:

- `08 96 01`: ключ `08` = поле 1, VARINT; `96 01` — число 150 в varint.
- `12 07 ...`: ключ `0x12` = поле 2, LEN; длина 7; байты строки в UTF-8.
- `-1` в `int32` занимает **10 байт** значения, а в `sint32` благодаря zigzag — один байт (`-1 → 1`, `1 → 2`).
- Список `nums` упакован: один ключ `2a` (поле 5, LEN), длина `04`, затем подряд `01 02 ac 02`.

## 2.3. Эволюция схемы

Смоделируем две версии сообщения. В v2 удалили поле `nickname` (номер и имя внесены в `reserved`) и добавили `email`. Разные пакеты `evolution.v1` и `evolution.v2` позволяют загрузить обе версии в один процесс, но для двоичного формата имена пакетов не важны: важны только номера и типы полей.

In [ ]:
%%writefile protos/evolution/v1/user.proto
syntax = "proto3";

package evolution.v1;

message User {
  int64 id = 1;
  string name = 2;
  string nickname = 3;
}

In [ ]:
%%writefile protos/evolution/v2/user.proto
syntax = "proto3";

package evolution.v2;

message User {
  reserved 3;
  reserved "nickname";
  int64 id = 1;
  string name = 2;
  string email = 4;     // новое поле с новым номером
}

In [ ]:
!{sys.executable} -m grpc_tools.protoc -I protos --python_out=. --pyi_out=. protos/evolution/v1/user.proto protos/evolution/v2/user.proto

from evolution.v1 import user_pb2 as user_v1
from evolution.v2 import user_pb2 as user_v2

# Прямая совместимость: старый код читает данные нового
new_bytes = user_v2.User(id=1, name="Ann", email="ann@example.com").SerializeToString()
old_view = user_v1.User.FromString(new_bytes)
print("v1 читает данные v2:", old_view)
print("email старому коду неизвестен, но сохранён:", old_view.SerializeToString() == new_bytes)
print("После пересылки через старый код:", user_v2.User.FromString(old_view.SerializeToString()))

# Обратная совместимость: новый код читает данные старого
old_bytes = user_v1.User(id=2, name="Bob", nickname="bobby").SerializeToString()
new_view = user_v2.User.FromString(old_bytes)
print("v2 читает данные v1:", new_view, "email =", repr(new_view.email))

Старый код не потерял новое поле `email`: он хранит его как **неизвестное** и записывает обратно при пересылке. Новый код получил для `email` значение по умолчанию.

Теперь пример **несовместимого** изменения: кто-то поменял тип поля 2 со `string` на `int32`. Для наглядности соберём такой класс из дескриптора прямо в коде, не создавая файл.

In [ ]:
from google.protobuf import descriptor_pb2, descriptor_pool, message_factory

FieldProto = descriptor_pb2.FieldDescriptorProto
broken_file = descriptor_pb2.FileDescriptorProto(name="broken.proto", package="broken", syntax="proto3")
broken_message = broken_file.message_type.add(name="User")
broken_message.field.add(name="id", number=1, type=FieldProto.TYPE_INT64, label=FieldProto.LABEL_OPTIONAL)
broken_message.field.add(name="name", number=2, type=FieldProto.TYPE_INT32, label=FieldProto.LABEL_OPTIONAL)

pool = descriptor_pool.DescriptorPool()
pool.Add(broken_file)
BrokenUser = message_factory.GetMessageClass(pool.FindMessageTypeByName("broken.User"))

broken = BrokenUser.FromString(user_v1.User(id=3, name="Eve").SerializeToString())
print("id =", broken.id, "| name =", broken.name)   # name == 0, ошибки нет!

Ошибки при разборе **нет**: байты строки не подошли к типу `int32` и ушли в неизвестные поля, а `name` получил значение по умолчанию 0. Такие ошибки трудно заметить, поэтому номера и типы полей — неизменяемая часть контракта.

---
# Часть 3. Первый gRPC-сервис: Hello World

## 3.1. Контракт

Сервис описывается в том же `.proto`, что и сообщения. Сообщения объявляются **на верхнем уровне файла**, а не внутри блока `service`.

In [ ]:
%%writefile protos/helloworld.proto
syntax = "proto3";

package helloworld;

// Сервис приветствий
service Greeter {
  rpc SayHello (HelloRequest) returns (HelloReply);
  rpc SayHelloAgain (HelloRequest) returns (HelloReply);
}

// Запрос: имя того, кого приветствуем
message HelloRequest {
  string name = 1;
}

// Ответ: текст приветствия
message HelloReply {
  string message = 1;
}

Теперь добавляем ключ `--grpc_python_out`: он создаёт `helloworld_pb2_grpc.py` с заглушкой клиента и базовым классом сервера.

In [ ]:
!{sys.executable} -m grpc_tools.protoc -I protos --python_out=. --pyi_out=. --grpc_python_out=. protos/helloworld.proto
!ls helloworld_pb2*

In [ ]:
import helloworld_pb2_grpc

print([name for name in dir(helloworld_pb2_grpc) if not name.startswith("_")])

- `GreeterStub` — заглушка для клиента: её методы выглядят как обычные функции, но отправляют запросы по сети;
- `GreeterServicer` — базовый класс для реализации сервиса на сервере;
- `add_GreeterServicer_to_server` — регистрирует реализацию на сервере;
- `Greeter` — экспериментальный API для вызовов без явного создания заглушки (нам не понадобится).

## 3.2. Сервер и клиент

In [ ]:
%%writefile greeter_server.py
"""Сервер gRPC: реализация сервиса helloworld.Greeter."""
import logging
from concurrent import futures

import grpc
import helloworld_pb2
import helloworld_pb2_grpc


class Greeter(helloworld_pb2_grpc.GreeterServicer):
    def SayHello(self, request, context):
        return helloworld_pb2.HelloReply(message=f"Hello, {request.name}!")

    def SayHelloAgain(self, request, context):
        return helloworld_pb2.HelloReply(message=f"Hello again, {request.name}!")


def serve(port: int = 50051) -> None:
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    helloworld_pb2_grpc.add_GreeterServicer_to_server(Greeter(), server)
    server.add_insecure_port(f"[::]:{port}")   # [::] — все сетевые интерфейсы
    server.start()                             # не блокирует
    print(f"Server started, listening on {port}", flush=True)
    server.wait_for_termination()              # блокирует до остановки процесса


if __name__ == "__main__":
    logging.basicConfig()
    serve()

In [ ]:
%%writefile greeter_client.py
"""Клиент gRPC для сервиса helloworld.Greeter."""
import logging

import grpc
import helloworld_pb2
import helloworld_pb2_grpc


def run() -> None:
    print("Will try to greet world ...")
    # Канал — соединение с сервером; with закрывает его по завершении
    with grpc.insecure_channel("localhost:50051") as channel:
        stub = helloworld_pb2_grpc.GreeterStub(channel)
        response = stub.SayHello(helloworld_pb2.HelloRequest(name="you"), timeout=5)
        print("Greeter client received:", response.message)
        response = stub.SayHelloAgain(helloworld_pb2.HelloRequest(name="you"), timeout=5)
        print("Greeter client received:", response.message)


if __name__ == "__main__":
    logging.basicConfig()
    run()

## 3.3. Запуск

В реальной работе сервер и клиент запускают в двух терминалах (из каталога `grpc_practice`):

```bash
python greeter_server.py      # терминал 1
python greeter_client.py      # терминал 2
```

Из ноутбука запустим сервер отдельным процессом, дождёмся готовности и вызовем клиента. Вывод сервера появится в терминале, из которого запущен Jupyter.

In [ ]:
import subprocess
import time

import grpc


def wait_for_grpc(address: str, timeout: float = 10) -> None:
    """Ждёт, пока gRPC-сервер начнёт принимать соединения."""
    with grpc.insecure_channel(address) as channel:
        grpc.channel_ready_future(channel).result(timeout=timeout)


greeter_process = subprocess.Popen([sys.executable, "greeter_server.py"])
wait_for_grpc("localhost:50051")
print("Сервер запущен, PID", greeter_process.pid)

In [ ]:
!{sys.executable} greeter_client.py

## 3.4. Расширяем сервис

Добавим метод `SayMyName` со своими сообщениями запроса и ответа. Порядок действий при любом изменении контракта:

1. изменить `.proto`;
2. перегенерировать код;
3. реализовать метод на сервере и перезапустить сервер;
4. вызвать метод в клиенте.

In [ ]:
%%writefile protos/helloworld.proto
syntax = "proto3";

package helloworld;

// Сервис приветствий
service Greeter {
  rpc SayHello (HelloRequest) returns (HelloReply);
  rpc SayHelloAgain (HelloRequest) returns (HelloReply);
  rpc SayMyName (MyNameRequest) returns (MyNameReply);
}

message HelloRequest {
  string name = 1;
}

message HelloReply {
  string message = 1;
}

// Сообщения нового метода — на верхнем уровне файла, не внутри service
message MyNameRequest {
  string name = 1;
}

message MyNameReply {
  string message = 1;
}

In [ ]:
!{sys.executable} -m grpc_tools.protoc -I protos --python_out=. --pyi_out=. --grpc_python_out=. protos/helloworld.proto

In [ ]:
%%writefile greeter_server.py
"""Сервер gRPC: реализация сервиса helloworld.Greeter."""
import logging
from concurrent import futures

import grpc
import helloworld_pb2
import helloworld_pb2_grpc


class Greeter(helloworld_pb2_grpc.GreeterServicer):
    def SayHello(self, request, context):
        return helloworld_pb2.HelloReply(message=f"Hello, {request.name}!")

    def SayHelloAgain(self, request, context):
        return helloworld_pb2.HelloReply(message=f"Hello again, {request.name}!")

    def SayMyName(self, request, context):
        return helloworld_pb2.MyNameReply(
            message=f"— You're {request.name}.\n— You're goddamn right."
        )


def serve(port: int = 50051) -> None:
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    helloworld_pb2_grpc.add_GreeterServicer_to_server(Greeter(), server)
    server.add_insecure_port(f"[::]:{port}")
    server.start()
    print(f"Server started, listening on {port}", flush=True)
    server.wait_for_termination()


if __name__ == "__main__":
    logging.basicConfig()
    serve()

In [ ]:
%%writefile greeter_client.py
"""Клиент gRPC для сервиса helloworld.Greeter."""
import logging

import grpc
import helloworld_pb2
import helloworld_pb2_grpc


def run() -> None:
    print("Will try to greet world ...")
    with grpc.insecure_channel("localhost:50051") as channel:
        stub = helloworld_pb2_grpc.GreeterStub(channel)
        response = stub.SayHello(helloworld_pb2.HelloRequest(name="you"), timeout=5)
        print("Greeter client received:", response.message)
        response = stub.SayHelloAgain(helloworld_pb2.HelloRequest(name="you"), timeout=5)
        print("Greeter client received:", response.message)
        # Аргумент — сообщение ЗАПРОСА MyNameRequest с полем name
        response = stub.SayMyName(helloworld_pb2.MyNameRequest(name="Heisenberg"), timeout=5)
        print(response.message)


if __name__ == "__main__":
    logging.basicConfig()
    run()

In [ ]:
# Перезапускаем сервер, чтобы он загрузил новый код
greeter_process.terminate()
greeter_process.wait()

greeter_process = subprocess.Popen([sys.executable, "greeter_server.py"])
wait_for_grpc("localhost:50051")
!{sys.executable} greeter_client.py

### Почему ошибка в типе аргумента может остаться незамеченной

Частая ошибка — передать в метод сообщение **ответа** вместо сообщения **запроса**: `stub.SayMyName(helloworld_pb2.MyNameReply(message="Heisenberg"))`. Проверим, что произойдёт.

In [ ]:
%%writefile wrong_type_client.py
"""Демонстрация ошибки: в метод передано сообщение ответа вместо сообщения запроса."""
import grpc
import helloworld_pb2
import helloworld_pb2_grpc

with grpc.insecure_channel("localhost:50051") as channel:
    stub = helloworld_pb2_grpc.GreeterStub(channel)
    # ОШИБКА: нужен MyNameRequest(name=...), а передан MyNameReply(message=...)
    wrong = stub.SayMyName(helloworld_pb2.MyNameReply(message="Heisenberg"), timeout=5)
    print(wrong.message)

In [ ]:
!{sys.executable} wrong_type_client.py

Вызов сработал — и это плохо. Двоичное сообщение не содержит имени типа: у `MyNameReply` поле 1 — строка `message`, у `MyNameRequest` поле 1 — строка `name`, байты совпадают. Если бы номера или типы полей различались, данные молча потерялись бы. Код с такой ошибкой вводит в заблуждение и сломается при первом изменении схемы, поэтому тип аргумента всегда сверяют с сигнатурой метода в `.proto`.

> Клиенты запускаются отдельными процессами, потому что в процесс ноутбука уже загружена старая версия `helloworld_pb2`. Сгенерированный модуль нельзя перезагрузить через `importlib.reload`: реестр описаний protobuf откажется принять изменённый файл с тем же именем. После изменения `.proto` в самом ноутбуке перезапускают ядро.

In [ ]:
greeter_process.terminate()
greeter_process.wait()
print("Сервер остановлен, код завершения", greeter_process.returncode)

---
# Часть 4. Потоковые вызовы

gRPC поддерживает четыре типа вызовов. Унарный мы уже видели; теперь реализуем три потоковых:

| Тип | В `.proto` | В Python на сервере | В Python на клиенте |
|---|---|---|---|
| Серверный поток | `returns (stream X)` | метод-генератор с `yield` | итерация по результату вызова |
| Клиентский поток | `(stream X) returns (Y)` | итерация по `request_iterator` | передать итератор или генератор |
| Двунаправленный | `(stream X) returns (stream Y)` | итерация по `request_iterator` + `yield` | итератор на входе, итерация по результату |

In [ ]:
%%writefile protos/stats.proto
syntax = "proto3";

package stats;

service Stats {
  // Серверный поток: обратный отсчёт от start до 0
  rpc Countdown (CountdownRequest) returns (stream CountdownReply);
  // Клиентский поток: сумма присланных чисел
  rpc Sum (stream Number) returns (SumReply);
  // Двунаправленный поток: скользящее среднее после каждого числа
  rpc RunningAverage (stream Number) returns (stream AverageReply);
}

message CountdownRequest { int32 start = 1; }
message CountdownReply   { int32 value = 1; }
message Number           { double value = 1; }
message SumReply         { double total = 1; int32 count = 2; }
message AverageReply     { double average = 1; int32 count = 2; }

In [ ]:
!{sys.executable} -m grpc_tools.protoc -I protos --python_out=. --pyi_out=. --grpc_python_out=. protos/stats.proto

In [ ]:
%%writefile stats_server.py
"""Сервер gRPC с потоковыми методами."""
import time
from concurrent import futures

import grpc
import stats_pb2
import stats_pb2_grpc


class StatsServicer(stats_pb2_grpc.StatsServicer):
    def Countdown(self, request, context):
        if request.start < 0:
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, "start must be >= 0")
        for value in range(request.start, -1, -1):
            if not context.is_active():          # клиент отменил вызов
                print("Countdown: клиент отключился", flush=True)
                return
            yield stats_pb2.CountdownReply(value=value)
            time.sleep(0.2)

    def Sum(self, request_iterator, context):
        total, count = 0.0, 0
        for number in request_iterator:          # читаем поток до конца
            total += number.value
            count += 1
        return stats_pb2.SumReply(total=total, count=count)

    def RunningAverage(self, request_iterator, context):
        total = 0.0
        for count, number in enumerate(request_iterator, start=1):
            total += number.value
            yield stats_pb2.AverageReply(average=total / count, count=count)


def serve(port: int = 50053) -> grpc.Server:
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    stats_pb2_grpc.add_StatsServicer_to_server(StatsServicer(), server)
    server.add_insecure_port(f"[::]:{port}")
    server.start()
    return server


if __name__ == "__main__":
    serve().wait_for_termination()

Метод `server.start()` не блокирует: сервер обрабатывает вызовы в фоновых потоках. Поэтому этот сервер можно запустить прямо в процессе ноутбука.

In [ ]:
import stats_pb2
import stats_pb2_grpc
import stats_server

stats_grpc_server = stats_server.serve(port=50053)
stats_channel = grpc.insecure_channel("localhost:50053")
stats_stub = stats_pb2_grpc.StatsStub(stats_channel)
print("Сервер Stats запущен")

In [ ]:
# Серверный поток: ответы приходят по мере готовности
start = time.monotonic()
for reply in stats_stub.Countdown(stats_pb2.CountdownRequest(start=5), timeout=10):
    print(f"{time.monotonic() - start:4.1f} с: {reply.value}")

In [ ]:
# Клиентский поток: отправляем генератор, получаем один ответ
def numbers(values):
    for value in values:
        yield stats_pb2.Number(value=value)


summary = stats_stub.Sum(numbers([1, 2, 3.5, 10]), timeout=5)
print(f"Сумма {summary.total} из {summary.count} чисел")

In [ ]:
# Двунаправленный поток: отправка и получение чередуются
def slow_numbers(values, delay=0.3):
    for value in values:
        print(f"  → отправляем {value}")
        yield stats_pb2.Number(value=value)
        time.sleep(delay)


for reply in stats_stub.RunningAverage(slow_numbers([10, 20, 30, 40]), timeout=10):
    print(f"  ← среднее после {reply.count} чисел: {reply.average}")

In [ ]:
# Отмена потока клиентом и ошибка INVALID_ARGUMENT
responses = stats_stub.Countdown(stats_pb2.CountdownRequest(start=20), timeout=10)
try:
    for reply in responses:
        print(reply.value, end=" ")
        if reply.value == 17:
            responses.cancel()                  # больше не нужно
except grpc.RpcError as error:
    print("\nВызов завершён:", error.code())

try:
    list(stats_stub.Countdown(stats_pb2.CountdownRequest(start=-1), timeout=5))
except grpc.RpcError as error:
    print("Ошибка:", error.code(), "—", error.details())

In [ ]:
stats_channel.close()
stats_grpc_server.stop(grace=None)
print("Сервер Stats остановлен")

---
# Часть 5. Ошибки, дедлайны, метаданные, асинхронный клиент

Опишем сервис библиотеки в **отдельном файле**, который импортирует сообщения из `library.proto`. Путь в `import` указывается от корня `-I protos`. Оба файла в одном пакете `library.v1`, поэтому к `Book` и `Genre` обращаемся по короткому имени.

In [ ]:
%%writefile protos/library/v1/library_service.proto
syntax = "proto3";

package library.v1;

import "library/v1/library.proto";

service LibraryService {
  // Унарный вызов: книга по идентификатору
  rpc GetBook (GetBookRequest) returns (Book);
  // Серверный поток: книги выбранного жанра (GENRE_UNSPECIFIED — все)
  rpc ListBooks (ListBooksRequest) returns (stream Book);
  // Долгая операция, доступная только с токеном
  rpc BuildReport (BuildReportRequest) returns (Report);
}

message GetBookRequest {
  int64 id = 1;
}

message ListBooksRequest {
  Genre genre = 1;
}

message BuildReportRequest {
  double work_seconds = 1;   // сколько секунд «работает» сервер — для демонстрации дедлайнов
}

message Report {
  int32 total_books = 1;
  map<string, int32> books_by_genre = 2;
}

In [ ]:
!{sys.executable} -m grpc_tools.protoc -I protos --python_out=. --pyi_out=. --grpc_python_out=. protos/library/v1/library.proto protos/library/v1/library_service.proto
!ls library/v1

In [ ]:
%%writefile library_server.py
"""Сервер LibraryService: коды ошибок, дедлайны, метаданные, reflection."""
import time
from collections import Counter
from concurrent import futures

import grpc
from grpc_reflection.v1alpha import reflection

from library.v1 import library_pb2, library_service_pb2, library_service_pb2_grpc

API_TOKEN = "secret-token"   # в реальном проекте — из переменной окружения или хранилища секретов


def make_books() -> dict[int, library_pb2.Book]:
    data = [
        (1, "Solaris", library_pb2.GENRE_FICTION, "Stanisław Lem"),
        (2, "A Brief History of Time", library_pb2.GENRE_SCIENCE, "Stephen Hawking"),
        (3, "The Guns of August", library_pb2.GENRE_HISTORY, "Barbara W. Tuchman"),
        (4, "Roadside Picnic", library_pb2.GENRE_FICTION, "Arkady and Boris Strugatsky"),
    ]
    books = {}
    for book_id, title, genre, author in data:
        book = library_pb2.Book(id=book_id, title=title, genre=genre)
        book.authors.add(name=author)
        books[book_id] = book
    return books


class LibraryService(library_service_pb2_grpc.LibraryServiceServicer):
    def __init__(self) -> None:
        self._books = make_books()

    def GetBook(self, request, context):
        metadata = dict(context.invocation_metadata())
        request_id = metadata.get("x-request-id", "-")
        # Завершающие метаданные клиент получит вместе с ответом или ошибкой
        context.set_trailing_metadata([("x-request-id", request_id), ("x-served-by", "library-1")])
        if request.id <= 0:
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, "id must be positive")
        book = self._books.get(request.id)
        if book is None:
            context.abort(grpc.StatusCode.NOT_FOUND, f"Book {request.id} not found")
        return book

    def ListBooks(self, request, context):
        for book in self._books.values():
            if request.genre in (library_pb2.GENRE_UNSPECIFIED, book.genre):
                yield book

    def BuildReport(self, request, context):
        metadata = dict(context.invocation_metadata())
        if metadata.get("authorization") != f"Bearer {API_TOKEN}":
            context.abort(grpc.StatusCode.UNAUTHENTICATED, "Missing or invalid token")
        finish_at = time.monotonic() + request.work_seconds
        while time.monotonic() < finish_at:        # имитация долгой работы
            if not context.is_active():            # дедлайн истёк или клиент отменил вызов
                print("BuildReport: клиент больше не ждёт, прекращаем работу", flush=True)
                return library_service_pb2.Report()
            time.sleep(0.05)
        by_genre = Counter(library_pb2.Genre.Name(book.genre) for book in self._books.values())
        return library_service_pb2.Report(total_books=len(self._books), books_by_genre=by_genre)


def serve(port: int = 50052) -> grpc.Server:
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    library_service_pb2_grpc.add_LibraryServiceServicer_to_server(LibraryService(), server)
    service_names = (
        library_service_pb2.DESCRIPTOR.services_by_name["LibraryService"].full_name,
        reflection.SERVICE_NAME,
    )
    reflection.enable_server_reflection(service_names, server)
    server.add_insecure_port(f"[::]:{port}")
    server.start()
    return server


if __name__ == "__main__":
    serve().wait_for_termination()

In [ ]:
import library_server
from library.v1 import library_service_pb2, library_service_pb2_grpc

library_grpc_server = library_server.serve(port=50052)
library_channel = grpc.insecure_channel("localhost:50052")
library_stub = library_service_pb2_grpc.LibraryServiceStub(library_channel)
print("Сервер LibraryService запущен")

## 5.1. Метаданные

Метаданные передаются в заголовках HTTP/2, а не в теле сообщения. Метод `with_call` возвращает кроме ответа объект вызова, из которого можно прочитать завершающие метаданные.

In [ ]:
book, call = library_stub.GetBook.with_call(
    library_service_pb2.GetBookRequest(id=1),
    metadata=[("x-request-id", "req-42")],
    timeout=2,
)
print(book.title, "—", book.authors[0].name)
print("Метаданные ответа:", dict(call.trailing_metadata()))

## 5.2. Коды статуса

Сервер завершает вызов с ошибкой через `context.abort(код, описание)`. На клиенте ошибка приходит исключением `grpc.RpcError` с методами `code()` и `details()`.

In [ ]:
for book_id in (2, 999, 0):
    try:
        found = library_stub.GetBook(library_service_pb2.GetBookRequest(id=book_id), timeout=2)
        print(f"id={book_id}: {found.title}")
    except grpc.RpcError as error:
        print(f"id={book_id}: {error.code().name} — {error.details()}")

In [ ]:
# Серверный поток с фильтром
for found in library_stub.ListBooks(library_service_pb2.ListBooksRequest(genre=library_pb2.GENRE_FICTION), timeout=5):
    print(found.id, found.title)

## 5.3. Аутентификация через метаданные и дедлайны

`BuildReport` требует токен в заголовке `authorization` и «работает» заданное число секунд. Проверим три сценария:

1. без токена — `UNAUTHENTICATED`;
2. с токеном, но дедлайн (0,5 с) меньше времени работы (2 с) — `DEADLINE_EXCEEDED`; сервер заметит, что клиент больше не ждёт, и прекратит работу;
3. с токеном и достаточным дедлайном — успех.

In [ ]:
auth = [("authorization", f"Bearer {library_server.API_TOKEN}")]
scenarios = [
    ("без токена", [], 1.0, 5),
    ("короткий дедлайн", auth, 2.0, 0.5),
    ("успех", auth, 1.0, 5),
]
for title, metadata, work_seconds, timeout in scenarios:
    started = time.monotonic()
    try:
        report = library_stub.BuildReport(
            library_service_pb2.BuildReportRequest(work_seconds=work_seconds),
            metadata=metadata,
            timeout=timeout,
        )
        result = f"OK: {report.total_books} книги, {dict(report.books_by_genre)}"
    except grpc.RpcError as error:
        result = f"{error.code().name} — {error.details()}"
    print(f"{title:<18} {time.monotonic() - started:4.1f} с  {result}")

## 5.4. Асинхронный клиент grpc.aio

С `grpc.aio` независимые вызовы выполняются параллельно. Jupyter поддерживает `await` на верхнем уровне ячейки; в обычном скрипте используйте `asyncio.run(main())`.

In [ ]:
import asyncio


async def fetch_many(ids: list[int]) -> None:
    async with grpc.aio.insecure_channel("localhost:50052") as channel:
        stub = library_service_pb2_grpc.LibraryServiceStub(channel)
        calls = [stub.GetBook(library_service_pb2.GetBookRequest(id=i), timeout=2) for i in ids]
        results = await asyncio.gather(*calls, return_exceptions=True)
    for book_id, result in zip(ids, results):
        if isinstance(result, grpc.aio.AioRpcError):
            print(f"id={book_id}: {result.code().name}")
        else:
            print(f"id={book_id}: {result.title}")


await fetch_many([1, 2, 3, 4, 42])

## 5.5. Отладка через grpcurl (по желанию)

Сервер включает **reflection**, поэтому инструмент `grpcurl` узнаёт схему у самого сервера. Установка: `brew install grpcurl` (macOS) или архив со страницы релизов github.com/fullstorydev/grpcurl (Linux). Пока сервер из ноутбука запущен, в терминале:

```bash
grpcurl -plaintext localhost:50052 list
grpcurl -plaintext localhost:50052 describe library.v1.LibraryService
grpcurl -plaintext -d '{"id": 1}' localhost:50052 library.v1.LibraryService/GetBook
grpcurl -plaintext -d '{"genre": "GENRE_FICTION"}' localhost:50052 library.v1.LibraryService/ListBooks
```

Данные вводятся и выводятся в JSON по тем же правилам, что и `json_format`.

In [ ]:
library_channel.close()
library_grpc_server.stop(grace=None)
print("Сервер LibraryService остановлен")

---
# Часть 6. Микросервисы: Recommendations и Marketplace

Соберём систему из двух сервисов:

```
Браузер / curl ──HTTP, REST, JSON──▶ Marketplace (FastAPI, порт 8000)
                                          │
                                          │ gRPC, Protobuf
                                          ▼
                                     Recommendations (порт 50051)
```

- **Recommendations** — внутренний сервис на gRPC: по категории возвращает случайные книги.
- **Marketplace** — веб-приложение на FastAPI: снаружи отдаёт REST API (с документацией OpenAPI на `/docs`) и HTML-страницу, а за рекомендациями обращается к Recommendations по gRPC через асинхронный клиент.

Это типичная схема из лекции: снаружи — REST, внутри — gRPC. Структура проекта:

```
microservices/
├── compose.yaml
├── .dockerignore
├── protos/
│   └── recommendations.proto      # общий контракт
├── recommendations/
│   ├── Dockerfile
│   ├── requirements.txt
│   ├── recommendations.py
│   └── test_recommendations.py
└── marketplace/
    ├── Dockerfile
    ├── requirements.txt
    ├── marketplace.py
    └── test_marketplace_integration.py
```

Контракт лежит в общем каталоге `protos/`, а код из него генерирует каждый сервис для себя.

In [ ]:
%%writefile microservices/protos/recommendations.proto
syntax = "proto3";

package recommendations;

enum BookCategory {
  BOOK_CATEGORY_UNSPECIFIED = 0;
  BOOK_CATEGORY_MYSTERY = 1;
  BOOK_CATEGORY_SCIENCE_FICTION = 2;
  BOOK_CATEGORY_SELF_HELP = 3;
}

message RecommendationRequest {
  int32 user_id = 1;
  BookCategory category = 2;
  int32 max_results = 3;
}

message BookRecommendation {
  int32 id = 1;
  string title = 2;
}

message RecommendationResponse {
  repeated BookRecommendation recommendations = 1;
}

service Recommendations {
  rpc Recommend (RecommendationRequest) returns (RecommendationResponse);
}

## 6.1. Сервис Recommendations

In [ ]:
%%writefile microservices/recommendations/recommendations.py
"""Сервис рекомендаций книг (gRPC)."""
import os
import random
from concurrent import futures

import grpc
import recommendations_pb2_grpc
from recommendations_pb2 import BookCategory, BookRecommendation, RecommendationResponse

BOOKS_BY_CATEGORY = {
    BookCategory.BOOK_CATEGORY_MYSTERY: [
        BookRecommendation(id=1, title="The Maltese Falcon"),
        BookRecommendation(id=2, title="Murder on the Orient Express"),
        BookRecommendation(id=3, title="The Hound of the Baskervilles"),
        BookRecommendation(id=4, title="The Name of the Rose"),
    ],
    BookCategory.BOOK_CATEGORY_SCIENCE_FICTION: [
        BookRecommendation(id=5, title="The Hitchhiker's Guide to the Galaxy"),
        BookRecommendation(id=6, title="Ender's Game"),
        BookRecommendation(id=7, title="Dune"),
        BookRecommendation(id=8, title="Solaris"),
    ],
    BookCategory.BOOK_CATEGORY_SELF_HELP: [
        BookRecommendation(id=9, title="The 7 Habits of Highly Effective People"),
        BookRecommendation(id=10, title="How to Win Friends and Influence People"),
        BookRecommendation(id=11, title="Man's Search for Meaning"),
    ],
}


class RecommendationService(recommendations_pb2_grpc.RecommendationsServicer):
    def Recommend(self, request, context):
        if request.max_results <= 0:
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, "max_results must be positive")
        if request.category == BookCategory.BOOK_CATEGORY_UNSPECIFIED:
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, "category is required")
        books = BOOKS_BY_CATEGORY.get(request.category)
        if books is None:
            context.abort(grpc.StatusCode.NOT_FOUND, f"Category {request.category} not found")
        num_results = min(request.max_results, len(books))
        return RecommendationResponse(recommendations=random.sample(books, num_results))


def create_server(port: int = 50051) -> tuple[grpc.Server, int]:
    """Создаёт сервер. port=0 — выбрать свободный порт (удобно в тестах)."""
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    recommendations_pb2_grpc.add_RecommendationsServicer_to_server(RecommendationService(), server)
    bound_port = server.add_insecure_port(f"[::]:{port}")
    return server, bound_port


if __name__ == "__main__":
    server, bound_port = create_server(int(os.getenv("PORT", "50051")))
    server.start()
    print(f"Recommendations service listening on {bound_port}", flush=True)
    server.wait_for_termination()

Тесты сервиса двух видов:

- **модульный** — вызываем метод класса напрямую, без сети; вместо контекста передаём `None` (подходит, только пока метод не обращается к `context`);
- **через настоящий сервер** — поднимаем сервер на свободном порту (`port=0`) и проверяем коды ошибок так, как их увидит клиент.

In [ ]:
%%writefile microservices/recommendations/test_recommendations.py
import grpc
import pytest
from recommendations import RecommendationService, create_server
from recommendations_pb2 import BookCategory, RecommendationRequest
from recommendations_pb2_grpc import RecommendationsStub


def test_recommend_returns_requested_number():
    service = RecommendationService()
    request = RecommendationRequest(
        user_id=1, category=BookCategory.BOOK_CATEGORY_MYSTERY, max_results=2
    )
    response = service.Recommend(request, context=None)
    assert len(response.recommendations) == 2


def test_recommend_is_limited_by_catalog_size():
    service = RecommendationService()
    request = RecommendationRequest(
        user_id=1, category=BookCategory.BOOK_CATEGORY_SELF_HELP, max_results=100
    )
    response = service.Recommend(request, context=None)
    assert len(response.recommendations) == 3


@pytest.fixture
def stub():
    server, port = create_server(port=0)
    server.start()
    with grpc.insecure_channel(f"localhost:{port}") as channel:
        yield RecommendationsStub(channel)
    server.stop(grace=None)


@pytest.mark.parametrize(
    ("category", "max_results", "expected_code"),
    [
        (BookCategory.BOOK_CATEGORY_MYSTERY, 0, grpc.StatusCode.INVALID_ARGUMENT),
        (BookCategory.BOOK_CATEGORY_UNSPECIFIED, 1, grpc.StatusCode.INVALID_ARGUMENT),
        (99, 1, grpc.StatusCode.NOT_FOUND),   # неизвестное значение enum допустимо в proto3
    ],
)
def test_errors(stub, category, max_results, expected_code):
    request = RecommendationRequest(user_id=1, category=category, max_results=max_results)
    with pytest.raises(grpc.RpcError) as exc_info:
        stub.Recommend(request, timeout=2)
    assert exc_info.value.code() == expected_code

In [ ]:
%%writefile microservices/recommendations/requirements.txt
grpcio
grpcio-tools
pytest

## 6.2. Приложение Marketplace на FastAPI

Ключевые решения:

- **Канал gRPC создаётся один раз** при старте приложения (`lifespan`) и закрывается при остановке. Создавать канал на каждый HTTP-запрос дорого.
- Используется **асинхронный клиент** `grpc.aio`, потому что обработчики FastAPI асинхронные: ожидание ответа Recommendations не блокирует обработку других запросов.
- Адрес сервиса берётся из **переменной окружения** `RECOMMENDATIONS_HOST`: локально это `localhost`, в Docker — имя сервиса `recommendations`.
- Каждый вызов с **дедлайном** 2 с. Ошибки gRPC превращаются в понятные HTTP-коды: `INVALID_ARGUMENT` → `400`, недоступность сервиса → `503`.

In [ ]:
%%writefile microservices/marketplace/marketplace.py
"""Marketplace: веб-приложение на FastAPI, получает рекомендации по gRPC."""
import html
import os
from contextlib import asynccontextmanager
from typing import Annotated, Literal

import grpc
from fastapi import FastAPI, HTTPException, Query, Request
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from recommendations_pb2 import BookCategory, RecommendationRequest
from recommendations_pb2_grpc import RecommendationsStub

RECOMMENDATIONS_HOST = os.getenv("RECOMMENDATIONS_HOST", "localhost")
RECOMMENDATIONS_PORT = os.getenv("RECOMMENDATIONS_PORT", "50051")

CategoryName = Literal["mystery", "science_fiction", "self_help"]
CATEGORIES = {
    "mystery": BookCategory.BOOK_CATEGORY_MYSTERY,
    "science_fiction": BookCategory.BOOK_CATEGORY_SCIENCE_FICTION,
    "self_help": BookCategory.BOOK_CATEGORY_SELF_HELP,
}


class BookOut(BaseModel):
    id: int
    title: str


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Один канал на всё время работы приложения
    channel = grpc.aio.insecure_channel(f"{RECOMMENDATIONS_HOST}:{RECOMMENDATIONS_PORT}")
    app.state.recommendations = RecommendationsStub(channel)
    yield
    await channel.close()


app = FastAPI(title="Marketplace", version="1.0.0", lifespan=lifespan)


async def fetch_recommendations(
    stub: RecommendationsStub, user_id: int, category: CategoryName, max_results: int
) -> list[BookOut]:
    request = RecommendationRequest(
        user_id=user_id, category=CATEGORIES[category], max_results=max_results
    )
    try:
        response = await stub.Recommend(request, timeout=2.0)
    except grpc.aio.AioRpcError as error:
        if error.code() == grpc.StatusCode.INVALID_ARGUMENT:
            raise HTTPException(status_code=400, detail=error.details()) from error
        raise HTTPException(status_code=503, detail="Recommendations service is unavailable") from error
    return [BookOut(id=book.id, title=book.title) for book in response.recommendations]


@app.get("/api/v1/recommendations", tags=["recommendations"])
async def get_recommendations(
    request: Request,
    user_id: int,
    category: CategoryName = "mystery",
    max_results: Annotated[int, Query(ge=1, le=10)] = 3,
) -> list[BookOut]:
    """Рекомендации книг выбранной категории."""
    return await fetch_recommendations(
        request.app.state.recommendations, user_id, category, max_results
    )


@app.get("/", response_class=HTMLResponse, include_in_schema=False)
async def homepage(request: Request) -> str:
    books = await fetch_recommendations(
        request.app.state.recommendations, user_id=1, category="mystery", max_results=3
    )
    items = "\n".join(f"      <li>{html.escape(book.title)}</li>" for book in books)
    return f"""<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <title>Online Books For You</title>
  </head>
  <body>
    <h1>Mystery books you may like</h1>
    <ul>
{items}
    </ul>
  </body>
</html>
"""

In [ ]:
%%writefile microservices/marketplace/test_marketplace_integration.py
"""Интеграционные тесты: нужны запущенные Marketplace и Recommendations."""
import json
import os
from urllib.request import urlopen

BASE_URL = os.getenv("MARKETPLACE_URL", "http://localhost:8000")


def test_homepage_renders_three_books():
    with urlopen(f"{BASE_URL}/", timeout=5) as response:
        page = response.read().decode("utf-8")
    assert "<title>Online Books For You</title>" in page
    assert page.count("<li>") == 3


def test_api_returns_recommendations():
    url = f"{BASE_URL}/api/v1/recommendations?user_id=1&category=science_fiction&max_results=2"
    with urlopen(url, timeout=5) as response:
        assert response.status == 200
        books = json.load(response)
    assert len(books) == 2
    assert set(books[0]) == {"id", "title"}

In [ ]:
%%writefile microservices/marketplace/requirements.txt
fastapi
uvicorn[standard]
grpcio
grpcio-tools
pytest

## 6.3. Локальный запуск без Docker

Сгенерируем код из общего контракта в каталог каждого сервиса и запустим оба сервиса отдельными процессами. Порт 50051 должен быть свободен: сервер Hello World из части 3 мы уже остановили.

In [ ]:
MS = Path("microservices")

for service in ("recommendations", "marketplace"):
    subprocess.run(
        [sys.executable, "-m", "grpc_tools.protoc", "-I", "../protos",
         "--python_out=.", "--pyi_out=.", "--grpc_python_out=.", "../protos/recommendations.proto"],
        cwd=MS / service,
        check=True,
    )
    print(service, "→", sorted(p.name for p in (MS / service).glob("recommendations_pb2*")))

In [ ]:
import requests

recommendations_process = subprocess.Popen([sys.executable, "recommendations.py"], cwd=MS / "recommendations")
wait_for_grpc("localhost:50051")

marketplace_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "marketplace:app", "--port", "8000"],
    cwd=MS / "marketplace",
    env={**os.environ, "RECOMMENDATIONS_HOST": "localhost"},
)


def wait_for_http(url: str, timeout: float = 15) -> None:
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            requests.get(url, timeout=1)
            return
        except requests.ConnectionError:
            time.sleep(0.3)
    raise TimeoutError(f"{url} не отвечает")


wait_for_http("http://127.0.0.1:8000/openapi.json")
print("Оба сервиса запущены. Документация API: http://127.0.0.1:8000/docs")

In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/api/v1/recommendations",
    params={"user_id": 1, "category": "science_fiction", "max_results": 2},
    timeout=5,
)
print(response.status_code, response.json())

# Валидация FastAPI: max_results вне диапазона → 422
bad = requests.get("http://127.0.0.1:8000/api/v1/recommendations",
                   params={"user_id": 1, "max_results": 50}, timeout=5)
print(bad.status_code, bad.json()["detail"][0]["msg"])

print(requests.get("http://127.0.0.1:8000/", timeout=5).text)

In [ ]:
# Модульные тесты Recommendations и интеграционные тесты Marketplace
!cd microservices/recommendations && {sys.executable} -m pytest -q
!cd microservices/marketplace && {sys.executable} -m pytest -q test_marketplace_integration.py

Что будет, если Recommendations недоступен? Остановим его и снова обратимся к Marketplace: веб-приложение не упадёт, а вернёт `503`.

In [ ]:
recommendations_process.terminate()
recommendations_process.wait()

response = requests.get("http://127.0.0.1:8000/api/v1/recommendations", params={"user_id": 1}, timeout=10)
print(response.status_code, response.json())

marketplace_process.terminate()
marketplace_process.wait()
print("Сервисы остановлены")

## 6.4. Контейнеризация

Каждый сервис — отдельный образ. Контекст сборки — каталог `microservices/`, чтобы в образ попал общий каталог `protos/`. Код из `.proto` генерируется **во время сборки образа**.

Порядок инструкций в Dockerfile подобран для кэширования слоёв: сначала копируем `requirements.txt` и ставим зависимости, потом копируем код. При изменении кода зависимости не переустанавливаются.

In [ ]:
%%writefile microservices/recommendations/Dockerfile
FROM python:3.12-slim

WORKDIR /service/recommendations

COPY recommendations/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY protos/ /service/protos/
COPY recommendations/ .
RUN python -m grpc_tools.protoc -I ../protos --python_out=. --pyi_out=. \
    --grpc_python_out=. ../protos/recommendations.proto

EXPOSE 50051
CMD ["python", "recommendations.py"]

In [ ]:
%%writefile microservices/marketplace/Dockerfile
FROM python:3.12-slim

WORKDIR /service/marketplace

COPY marketplace/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY protos/ /service/protos/
COPY marketplace/ .
RUN python -m grpc_tools.protoc -I ../protos --python_out=. --pyi_out=. \
    --grpc_python_out=. ../protos/recommendations.proto

EXPOSE 8000
CMD ["uvicorn", "marketplace:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile microservices/.dockerignore
**/__pycache__
**/.pytest_cache
**/*_pb2.py
**/*_pb2.pyi
**/*_pb2_grpc.py

`.dockerignore` не даёт скопировать в образ локально сгенерированный код и кэш: в образе код генерируется заново из контракта.

### Запуск контейнеров вручную

Команды выполняются в терминале из каталога `grpc_practice/microservices`.

```bash
docker build -f recommendations/Dockerfile -t recommendations .
docker build -f marketplace/Dockerfile -t marketplace .
```

Если запустить два контейнера по отдельности, Marketplace **не найдёт** Recommendations по адресу `localhost:50051`: внутри контейнера `localhost` — это сам контейнер. Нужна общая сеть Docker, в которой контейнеры обращаются друг к другу по имени:

```bash
docker network create microservices

docker run -d --rm --network microservices --name recommendations recommendations

docker run -d --rm --network microservices --name marketplace \
    -e RECOMMENDATIONS_HOST=recommendations \
    -p 127.0.0.1:8000:8000 marketplace

curl "http://127.0.0.1:8000/api/v1/recommendations?user_id=1"

docker stop marketplace recommendations
docker network rm microservices
```

Порт Recommendations наружу не публикуется (`-p` не указан): сервис внутренний, к нему обращается только Marketplace.

## 6.5. Docker Compose

Compose описывает всю систему в одном файле. Сеть создаётся автоматически, имя сервиса служит DNS-именем. Ключ `version:` в современном формате Compose не нужен.

In [ ]:
%%writefile microservices/compose.yaml
services:
  recommendations:
    build:
      context: .
      dockerfile: recommendations/Dockerfile
    image: recommendations

  marketplace:
    build:
      context: .
      dockerfile: marketplace/Dockerfile
    image: marketplace
    environment:
      RECOMMENDATIONS_HOST: recommendations
    ports:
      - "127.0.0.1:8000:8000"
    depends_on:
      - recommendations

В терминале из каталога `grpc_practice/microservices`:

```bash
docker compose up --build -d          # собрать образы и запустить в фоне
docker compose ps                     # состояние сервисов
docker compose logs -f marketplace    # логи (Ctrl+C — выйти)

# тесты внутри контейнеров
docker compose exec recommendations pytest -q
docker compose exec marketplace pytest -q test_marketplace_integration.py

docker compose down                   # остановить и удалить контейнеры и сеть
```

`depends_on` гарантирует только порядок **запуска** контейнеров, а не готовность сервиса. Marketplace справляется с этим сам: пока Recommendations не готов, запросы получат `503`, а канал gRPC переподключится автоматически.

---
# Итоги

- Protobuf — формат сериализации: схема в `.proto`, код генерируется `grpc_tools.protoc`, в сообщении передаются номера полей и компактно закодированные значения.
- Путь Python-модуля определяется расположением `.proto` относительно `-I`, а не директивой `package`.
- Номера и типы полей — неизменяемая часть контракта; удалённые поля вносят в `reserved`. Несовместимые изменения не вызывают ошибок, а тихо портят данные.
- gRPC добавляет к protobuf сервисы, генерацию заглушек, HTTP/2, потоковые вызовы, коды статуса, дедлайны и метаданные.
- В каждом вызове задавайте `timeout`; ожидаемые ошибки возвращайте через `context.abort` с подходящим кодом.
- Типичная архитектура: снаружи REST (FastAPI с OpenAPI), внутри gRPC; сервисы находят друг друга по именам в сети Docker.

## Упражнения для самопроверки

1. Добавьте в `Greeter` метод `SayGoodbye` с собственными сообщениями запроса и ответа. Перегенерируйте код, реализуйте метод и вызовите его из клиента.
2. Сколько байтов займёт сообщение `Demo(a=300)`? Проверьте ответ, вычислив его вручную по правилам varint.
3. Добавьте в `RecommendationRequest` поле `repeated int32 exclude_ids = 4;`, чтобы не рекомендовать уже прочитанные книги. Проверьте, что старый Marketplace (без перегенерации кода) продолжает работать с новым Recommendations.
4. Реализуйте в `LibraryService` клиентский поток `AddBooks (stream Book) returns (Report)`.
5. Добавьте в Marketplace параметр `category` для главной страницы и тест, проверяющий ответ `400` при ошибке `INVALID_ARGUMENT` от Recommendations.